This notebooks aim is to answer the question, which terms actually exist in the picrust2 results of the data. Is there more terms that we are missing? and How to group these terms

# 1. Imports and Data preparation

In [19]:
import os
import sys
from pathlib import Path
import pickle
import pyarrow

# Data processing and analysis
import pandas as pd
import numpy as np
import re
from collections import Counter, defaultdict
from typing import Set, List, Union, Dict, Tuple, Optional

In [20]:

sys.path.append(os.path.abspath('..'))  # Ensures the project root is in Python's search path

if Path("/kaggle").exists():
    
    # Create directory structure
    !mkdir -p corrosion_scoring
    
    # Download the necessary files each session always
    !wget -O corrosion_scoring/__init__.py https://raw.githubusercontent.com/MagicAlex238/2_Micro/main/corrosion_scoring_root/corrosion_scoring/__init__.py
    !wget -O corrosion_scoring/global_terms.py https://raw.githubusercontent.com/MagicAlex238/2_Micro/main/corrosion_scoring_root/corrosion_scoring/global_terms.py
    !wget -O corrosion_scoring/scoring_system.py https://raw.githubusercontent.com/MagicAlex238/2_Micro/main/corrosion_scoring_root/corrosion_scoring/scoring_system.py
    !wget -O corrosion_scoring/term_processor.py https://raw.githubusercontent.com/MagicAlex238/2_Micro/main/corrosion_scoring_root/corrosion_scoring/term_processor.py
    # Add current directory to path
    import sys
    sys.path.append(os.getcwd())
    
    # Import package
    import corrosion_scoring as cs
else:
    print("Running in local (VSCode) environment")
    
    ## When in vscode local env first time only
    #!pip install git+https://github.com/MagicAlex238/2_Micro.git#subdirectory=corrosion_scoring_root
    import corrosion_scoring as cs

Running in local (VSCode) environment


In [65]:
# environment check
is_kaggle = os.path.exists("/kaggle")
if is_kaggle:
    # For Kaggle # Whole filtered Data
    base_dir =  Path("/kaggle/input/")
    eccontri_path = base_dir / "/eccontri-uniprot-enriched/ECcontri_Uniprot_enriched.parquet"
    pathway_path = base_dir / "gterms/pathways.csv"
    react_path = base_dir / "gterms/reactions.csv"
    # Directory to output large files 
    large_dir =  Path("/kaggle/working/")
    # Directory to output large files # eccontris, compilated db

else:              
    # For Vscode # Whole filtered Data
    # large galaxies input and output #large size dir for large files hosted instead in kaggle
    large_dir = Path("/home/beatriz/MIC")
    # Directory to output large files # eccontris, compilated dbs
    output_large = large_dir / "output_large"
    # Whole filtered Data
    eccontri_path = output_large / 'ECcontri_Uniprot_enriched.parquet'
    pathway_path = large_dir / "2_Micro/data_picrust/pathways.csv"
    react_path = large_dir / "2_Micro/data_picrust/reactions.csv"

In [22]:
# Whole filtered Data
ECcontri_Uniprot_enriched = pd.read_parquet(eccontri_path)

In [66]:
# Load list pathways
pathways = pd.read_csv(pathway_path, header=None)[0].tolist()
reactions = pd.read_csv(react_path, header=None)[0].tolist()

# 2. Validating terms as real_terms
## 2.1. Checking the terms agains the teoretical global_terms

In [67]:
def validate_terms(df, global_terms_list):
    """
    Checks which terms from a list of global dictionaries exist in the data.
    
    Args: df : dataframe with data to check
          global_terms: global list of dictionaries with metal_terms, pathways, mechanisms, functional categories, organic proceses and keywords
    
    Returns:  dict: {category: [found_terms]}
    """
    df = df.copy()
    found = {}
    cols_terms = ['enzyme_class', 'pathways', 'hierarchy', 'metals_consolidated',
            'corrosion_mechanisms', 'functional_categories', 
            'corrosion_keyword_groups', 'corrosion_synergies', 'organic_processes'
        ]

    for d in global_terms_list:
        for category, terms in d.items():
            if isinstance(terms, dict):
                # Handle functional_categories special case, nested with scores
                if 'terms' in terms and 'score' in terms:
                    # This is functional_categories format: {'terms': [...], 'score': 1.5}
                    existing = []
                    for term in terms['terms']:  # Access the 'terms' key
                        for col in cols_terms:
                            if col in df.columns:
                                if df[col].fillna('').astype(str).str.contains(term, case=False, regex=False).any():
                                    existing.append(term)
                                    break
                    if existing:
                        found[category] = existing
                else:
                    # Handle other nested dictionaries
                    for subcategory, subterms in terms.items():
                        if isinstance(subterms, list):  # Making sure it's a list
                            existing = []
                            for term in subterms:
                                for col in cols_terms:
                                    if col in df.columns:
                                        if df[col].fillna('').astype(str).str.contains(term, case=False, regex=False).any():
                                            existing.append(term)
                                            break
                            if existing:
                                found[f"{category}.{subcategory}"] = existing
            else:
                # Handle simple lists
                existing = []
                for term in terms:
                    for col in cols_terms:
                        if col in df.columns:
                            if df[col].fillna('').astype(str).str.contains(term, case=False, regex=False).any():
                                existing.append(term)
                                break
                if existing:
                    found[category] = existing
    
    return found
#sample= ECcontri_Uniprot_enriched.sample(n=15000)

In [25]:
sample =ECcontri_Uniprot_enriched.sample(n=15000)

In [26]:
real_terms = validate_terms(sample,
    [cs.metal_terms,
    cs.corrosion_mechanisms,
    cs.pathway_categories,
    cs.organic_categories,
    cs.corrosion_synergies,
    cs.functional_categories,
    cs.corrosion_keyword_groups
])

In [68]:
# path to the list of dictionaries
is_kaggle = os.path.exists("/kaggle")
if is_kaggle:
    rt_path = large_dir / 'real_terms.pkl'
    gterms_path = large_dir / 'real_pathways_reactions.pkl'
else:
    rt_path = output_large / 'real_terms.pkl'
    gterms_path = output_large / 'real_pathways_reactions.pkl'

In [ ]:
# Saving the new dataframe
#with open(rt_path, 'wb') as f:
#    pickle.dump(real_terms, f)

In [ ]:
# Reading the dictionaries
with open(rt_path, 'rb') as f:
    real_terms= pickle.load(f)
# Print in compact format
for category, terms in real_terms.items():
    terms_str = ', '.join(terms)
    print(f"'{category}': [{terms_str}]")

'iron': [Fe3+, iron, ferric, heme, iron-sulfur, siderophore, ferritin, ferredoxin, rubredoxin, iron-sulfur cluster]
'manganese': [manganese, mn]
'copper': [Cu+, copper, cupric]
'nickel': [Ni2+, nickel]
'cobalt': [cobalt, cobalamin, vitamin B12]
'magnesium': [magnesium]
'calcium': [Ca2+, calcium]
'Mo': [Mo, molybdenum, molybdopterin, molybdenum cofactor, molybdate]
'V5+': [V5+, vanadium]
'Al3+': [Al3+]
'Cr3+': [Cr3+, chromate]
'zinc': [Zn2+, zinc]
'sodium': [sodium]
'potassium': [potassium]
'selenium': [selenium, Se, selenocysteine, selenoprotein, selenite, selenate]
'lead': [lead]
'arsenic': [arsenic, arsenite, arsenate]
'mercury': [mercury, mercuric]
'phosphate': [phosphate, orthophosphate]
'nitrate': [NO3-, nitrate]
'nitrite': [nitrite]
'chloride': [Cl-, chloride]
'sulfate': [sulfate]
'sulfide': [sulfide, desulfovibrio, h2s]
'thiosulfate': [thiosulfate]
'oxygen': [O2, oxygen, oxidase, superoxide, peroxide]
'hydrogen': [hydrogenase, h2]
'organics': [methane, methane, methanogenesis, f

# Pathways
The original names of the pathways on the picrust results are given on a list and from this list, it would be run the same function to see which of this pathway names are captured on the list as real terms. Same with reactions

In [69]:
# for 2 dictionaries 
gterms = [{"pathways": pathways}, {"reactions": reactions}]

In [ ]:
real_pathways_reactions = validate_terms(ECcontri_Uniprot_enriched, gterms)
with open(gterms_path, 'wb') as f:
    pickle.dump(real_pathways_reactions, f)

In [ ]:
# Reading the dictionaries
with open(gterms_path, 'rb') as f:
    real_pathways_reactions= pickle.load(f)
# Print in compact format
for category, terms in real_pathways_reactions.items():
    terms_str = ', '.join(terms)
    print(f"'{category}': [{terms_str}]")

## 2.2 Critical review of real terms agains global terms

The process undergone for the scoring system has been iterative and during the first iteration it was noticed that pathways and mechanisms are highly interconnected due to the fact that one bacterium expresses multiple proteins across many pathways and mechanisms. Pathway and mechanism categories exhibit substantial overlap due to multi-protein expression patterns within individual bacterial strains. As a response to this iteration, functional_categories were designed to reconcile this complexity by grouping related processes and make it more about functional metabolism. On a following iteration more modern terms were introduced and diverse terms were tried. Ultimately, it was evident that it was necesary a reality check to validate the terms with the bioinformatics annotations. 
An script was done to critically evaluate the real terms possible to be mined from the compiled database after the enrichment of the data (ECcontri_Uniprot_enriched) with ec_records. The script compared the enriched data with the global terms which teoretically proposed the dictionaries grouped by categories namely: metal_terms, mechanisms, pathways, functional_categories, organic_processes, synergies and keywords.
Analysis of the enriched data's real terms, prompt to redefine the corrosion scoring system by eliminating non-existent terms and consolidating overlapping categorical structures. Yet some theoretical terms are left for teoretical completness. A manual curation was done to reasign categories for efficient computational resource allocation.
The categories to consolidate are: corrosion_synergies,metal_terms, functional_categories, mechanisms and pathways. The categories to remove are: corrosion_keyword_groups and organic_processes. A hierarchy of the categories is stablished, this allows a first term algorithm to prioritize the terms allocated, in order to prevent duplicates.
The scoring takes into account only corrosion_synergies,metal_terms and functional_categories.


# 2.3 Finding new real_terms : class BiologicalTermDiscovery
Before the terms are compile and arrange on different dictionaries, it was created a script to assest the possible new terms that could be in the enriched df. By using a comprehensive toolkit for discovering and analyzing biological terms from datasets. This class provides methods to extract terms from text data, identify patterns,and discover novel biological terms based on existing validated terms (real_terms).

In [72]:
class BiologicalTermDiscovery:
    """
    A comprehensive toolkit for discovering and analyzing biological terms from datasets.
    
    This class provides methods to extract terms from text data, identify patterns,
    and discover novel biological terms based on existing validated terms.
    """
    
    def __init__(self, min_term_length: int = 3, stop_words: Optional[Set[str]] = None):
        """
        Initialize the BiologicalTermDiscovery class.
        
        Args:
            min_term_length: Minimum length for extracted terms
            stop_words: Set of words to exclude from analysis
        """
        self.min_term_length = min_term_length
        self.stop_words = stop_words or {
            'and', 'or', 'the', 'of', 'in', 'to', 'for', 'with', 'by', 
            'from', 'at', 'on', 'high', 'general', 'families', 'viral', 
            'rna', 'gene', 'direct', 'organics', 'groups', 'ambiguous', 'messenger'
        }
    
    def extract_all_terms(self, series: pd.Series) -> Counter:
        """
        Extract all terms from a pandas Series containing text data.
        
        Args:
            series: Pandas Series containing text data
            
        Returns:
            Counter object with term frequencies
        """
        terms = Counter()
        
        for text in series.dropna():
            if isinstance(text, str):
                # Split on various separators while preserving meaningful phrases
                chunks = re.split(r'[,;|\n\t\s\(\)\[\]]+', text)
                
                for chunk in chunks:
                    # Clean but preserve meaningful separators
                    cleaned = re.sub(r'^[^\w]+|[^\w]+$', '', chunk.lower())
                    
                    # Filter terms based on length and stop words
                    if (len(cleaned) >= self.min_term_length and 
                        not cleaned.isdigit() and 
                        cleaned not in self.stop_words):
                        terms[cleaned] += 1
                        
                        # Extract individual words from compound terms
                        if '_' in cleaned or '-' in cleaned:
                            parts = re.split(r'[_-]+', cleaned)
                            for part in parts:
                                if (len(part) >= self.min_term_length and 
                                    not part.isdigit() and 
                                    part not in self.stop_words):
                                    terms[part] += 1
        
        return terms
    
    def create_pattern_dictionary(self, real_terms_dict: Dict[str, List[str]]) -> Dict[str, List[str]]:
        """
        Create pattern dictionary from real terms to identify similar terms.
        
        Args:  real_terms_dict: Dictionary mapping categories to lists of real terms
            
        Returns: Dictionary mapping patterns to example terms
        """
        all_terms = []
        for category, terms_list in real_terms_dict.items():
            all_terms.extend(terms_list)
        
        pattern_groups = defaultdict(set)
        
        for term in all_terms:
            term_lower = term.lower()
            
            # Extract meaningful prefixes
            if len(term_lower) >= 5:
                for prefix_len in [3, 4, 5]:
                    if prefix_len < len(term_lower):
                        prefix = term_lower[:prefix_len]
                        pattern_groups[prefix].add(term_lower)
            
            # Extract meaningful suffixes
            if len(term_lower) >= 6:
                for suffix_len in [3, 4]:
                    suffix = term_lower[-suffix_len:]
                    if suffix in ['ase', 'tion', 'ate', 'ose', 'ine']:
                        pattern_groups[f"*{suffix}"].add(term_lower)
        
        # Keep only patterns with multiple matches
        return {
            pattern: list(terms) for pattern, terms in pattern_groups.items() 
            if len(terms) >= 2
        }
    
    def find_pattern_matches(self, all_terms: Counter, patterns: Dict[str, List[str]]) -> Dict[str, List[Tuple[str, int]]]:
        """
        Find terms that match established patterns but are not in the original real set.
        
        Args: all_terms: Counter of all extracted terms
            patterns: Dictionary of patterns to match against
            
        Returns: Dictionary mapping pattern types to lists of (term, frequency) tuples
        """
        pattern_discoveries = defaultdict(list)
        
        for pattern, known_examples in patterns.items():
            known_set = set(known_examples)
            
            if pattern.startswith('*'):
                # Suffix pattern matching
                suffix = pattern[1:]
                for term, freq in all_terms.items():
                    if term.endswith(suffix) and term not in known_set:
                        pattern_discoveries[f"new_{suffix}_terms"].append((term, freq))
            else:
                # Prefix pattern matching
                for term, freq in all_terms.items():
                    if term.startswith(pattern) and term not in known_set:
                        pattern_discoveries[f"new_{pattern}_variants"].append((term, freq))
        
        return pattern_discoveries
    
    def categorize_biological_discoveries(self, pattern_matches: Dict[str, List[Tuple]], 
                                        novel_terms: List[Tuple]) -> Dict[str, List[Tuple]]:
        """
        Categorize discoveries into biological categories - with deduplication.
        
        Args:  pattern_matches: Dictionary of pattern matches
               novel_terms: List of novel terms with frequencies
            
        Returns: Dictionary mapping biological categories to relevant discoveries
        """
        categories = {
            'metal_related': [],
            'enzymes': [],
            'metabolic_processes': [],
            'chemical_compounds': []
        }
        
        # Track terms we've already categorized to avoid duplicates
        seen_terms = set()
        
        # Categorize pattern matches
        for pattern, matches in pattern_matches.items():
            for term, freq in matches:
                if term in seen_terms:
                    continue  # Skip if we've already categorized this term
                    
                # Better categorization rules - enzymes get priority
                if term.endswith('ase') or 'ase' in term or any(enzyme in term for enzyme in ['reductase', 'oxidase', 'transferase', 'hydrolase', 'lyase', 'ligase']):
                    categories['enzymes'].append((term, freq, pattern))
                    seen_terms.add(term)
                elif any(process in term for process in ['metabolism', 'synthesis', 'production', 'reduction', 'oxidation', 'transfer']):
                    categories['metabolic_processes'].append((term, freq, pattern))
                    seen_terms.add(term)
                elif any(metal in term for metal in ['fer', 'iron', 'copper', 'zinc', 'cobalt', 'manganese', 'nickel']):
                    categories['metal_related'].append((term, freq, pattern))
                    seen_terms.add(term)
                elif any(chem in term for chem in ['acid', 'amine', 'ate', 'ose']) and not any(enzyme in term for enzyme in ['ase']):
                    categories['chemical_compounds'].append((term, freq, pattern))
                    seen_terms.add(term)
        
        # Sort each category by frequency and take top 5
        for category in categories:
            categories[category] = sorted(categories[category], 
                                        key=lambda x: x[1], reverse=True)[:5]
        
        return categories
    
    def discover_novel_terms(self, df: pd.DataFrame, 
                           real_terms_dict: Dict[str, List[str]],
                           text_columns: List[str],
                           min_frequency: int = 10,
                           top_n_per_pattern: int = 5,
                           min_pattern_novelty: int = 50) -> Dict:
        """
        Main method to discover novel biological terms using pattern analysis.
        
        Args:
            df: DataFrame containing the biological data
            real_terms_dict: Dictionary of real terms by category
            text_columns: List of column names containing text data
            min_frequency: Minimum frequency threshold for terms
            top_n_per_pattern: Maximum number of terms to return per pattern
            min_pattern_novelty: Minimum frequency for completely novel terms
            
        Returns:
            Comprehensive dictionary of discoveries and analysis results
        """
        # Extract all terms from specified columns
        all_discovered_terms = Counter()
        processed_columns = []
        
        for col in text_columns:
            if col in df.columns:
                processed_columns.append(col)
                column_terms = self.extract_all_terms(df[col])
                all_discovered_terms.update(column_terms)
        
        # Create pattern dictionary from real terms
        patterns = self.create_pattern_dictionary(real_terms_dict)
        
        # Focus on biologically relevant patterns
        biological_prefixes = {'fer', 'iron', 'sulf', 'thio', 'oxy', 'nitr', 'met', 'acet', 'prop'}
        enzyme_suffixes = {'*ase', '*tion'}
        
        focused_patterns = {}
        for pattern, examples in patterns.items():
            if (any(pattern.startswith(prefix) for prefix in biological_prefixes) or
                any(pattern == suffix for suffix in enzyme_suffixes) or
                len(examples) >= 3):
                focused_patterns[pattern] = examples
        
        # Create flat set of all real terms
        all_real_terms = set()
        for terms_list in real_terms_dict.values():
            all_real_terms.update(term.lower() for term in terms_list)
        
        # Find pattern-based discoveries
        pattern_matches = self.find_pattern_matches(all_discovered_terms, focused_patterns)
        
        # Filter by frequency and biological relevance - with KEGG code filtering
        filtered_pattern_matches = {}
        for pattern, matches in pattern_matches.items():
            high_freq_matches = [
                (term, freq) for term, freq in matches 
                if freq >= min_frequency and 
                not any(skip_word in term for skip_word in self.stop_words) and
                not term.startswith('br:ko') and  # Filter out KEGG codes
                not re.match(r'^[a-z]{2}:\w+\d+', term)  # Filter out database IDs
            ]
            
            if high_freq_matches:
                filtered_pattern_matches[pattern] = sorted(high_freq_matches, 
                                                         key=lambda x: x[1], 
                                                         reverse=True)[:top_n_per_pattern]
        
        # Find completely novel high-frequency terms - with KEGG code filtering
        novel_high_freq = []
        for term, freq in all_discovered_terms.most_common(50):
            if (freq >= min_pattern_novelty and
                term not in all_real_terms and
                len(term) >= 5 and
                not any(skip_word in term for skip_word in self.stop_words) and
                not term.startswith('br:ko') and  # Filter out KEGG codes
                not re.match(r'^[a-z]{2}:\w+\d+', term) and  # Filter out database IDs
                not any(term.startswith(p) or (p.startswith('*') and term.endswith(p[1:])) 
                       for p in focused_patterns.keys())):
                novel_high_freq.append((term, freq))
        
        # Categorize biological discoveries
        biological_categories = self.categorize_biological_discoveries(
            filtered_pattern_matches, novel_high_freq
        )
        
        return {
            'discovery_summary': {
                'total_unique_terms': len(all_discovered_terms),
                'real_terms_baseline': len(all_real_terms),
                'focused_patterns_used': len(focused_patterns),
                'pattern_matches_found': sum(len(matches) for matches in filtered_pattern_matches.values()),
                'novel_high_freq_terms': len(novel_high_freq),
                'columns_processed': processed_columns,
                'real_categories': list(real_terms_dict.keys()),
                'filtering_parameters': {
                    'min_frequency': min_frequency,
                    'min_novelty': min_pattern_novelty,
                    'top_n_per_pattern': top_n_per_pattern
                }
            },
            'focused_patterns': focused_patterns,
            'pattern_discoveries': filtered_pattern_matches,
            'novel_high_frequency': novel_high_freq[:10],
            'pattern_summary': {
                pattern: len(matches) for pattern, matches in filtered_pattern_matches.items()
            },
            'biological_categories': biological_categories
        }
    
    def generate_discovery_report(self, df: pd.DataFrame, 
                                real_terms_dict: Dict[str, List[str]],
                                text_columns: List[str],
                                focused: bool = True) -> Dict:
        """
        Generate a comprehensive discovery report.
        
        Args:
            df: DataFrame containing the biological data
            real_terms_dict: Dictionary of real terms by category
            text_columns: List of column names containing text data
            focused: Whether to use focused (stricter) parameters
            
        Returns:
            Dictionary containing the discovery results
        """
        if focused:
            discoveries = self.discover_novel_terms(
                df, real_terms_dict, text_columns,
                min_frequency=10,
                top_n_per_pattern=3,
                min_pattern_novelty=100
            )
        else:
            discoveries = self.discover_novel_terms(
                df, real_terms_dict, text_columns,
                min_frequency=5,
                top_n_per_pattern=10,
                min_pattern_novelty=20
            )
        
        return discoveries
    
    def print_discovery_report(self, discoveries: Dict, title: str = "BIOLOGICAL TERM DISCOVERY REPORT") -> None:
        """
        Print a formatted discovery report.
        
        Args:
            discoveries: Dictionary returned by generate_discovery_report
            title: Title for the report
        """
        print(f"🔬 {title} 🔬")
        print("=" * 60)
        
        summary = discoveries['discovery_summary']
        params = summary['filtering_parameters']
        
        print(f"📊 Analysis Parameters:")
        print(f"   • Minimum frequency: {params['min_frequency']}")
        print(f"   • Minimum novelty threshold: {params['min_novelty']}")
        print(f"   • Top terms per pattern: {params['top_n_per_pattern']}")
        
        print(f"\n📈 Discovery Statistics:")
        print(f"   • Total unique terms found: {summary['total_unique_terms']:,}")
        print(f"   • Real terms baseline: {summary['real_terms_baseline']:,}")
        print(f"   • Focused patterns analyzed: {summary['focused_patterns_used']}")
        print(f"   • High-quality pattern matches: {summary['pattern_matches_found']}")
        print(f"   • Novel high-frequency terms: {summary['novel_high_freq_terms']}")
        
        print(f"\n🧬 BIOLOGICAL DISCOVERIES BY CATEGORY:")
        bio_categories = discoveries['biological_categories']
        
        for category, items in bio_categories.items():
            if items:
                print(f"\n  🔹 {category.upper().replace('_', ' ')}:")
                for item in items:
                    if len(item) == 3:  # Pattern match
                        term, freq, pattern = item
                        print(f"     • {term} ({freq}x) [pattern: {pattern}]")
                    else:  # Novel term
                        term, freq = item
                        print(f"     • {term} ({freq}x)")
        
        if discoveries['novel_high_frequency']:
            print(f"\n⭐ TOP NOVEL HIGH-FREQUENCY TERMS:")
            for term, freq in discoveries['novel_high_frequency']:
                print(f"   • {term} ({freq}x)")
        
        print("=" * 60)


# Convenience function to analyze a biological dataset
def analyze_biological_dataset(df: pd.DataFrame, 
                             real_terms_dict: Dict[str, List[str]],
                             text_columns: List[str]) -> Tuple[Set[str], Dict]:
    """
    Convenience function to analyze a biological dataset and extract new terms.
    
    Args:
        df: DataFrame containing biological data
        real_terms_dict: Dictionary of real terms by category
        text_columns: List of column names containing text data
        
    Returns:
        Tuple of (new_terms_set, discoveries_dict)
    """
    analyzer = BiologicalTermDiscovery()
    discoveries = analyzer.generate_discovery_report(df, real_terms_dict, text_columns)
    
    # Extract all discovered terms as a set
    new_terms = set()
    for matches in discoveries['pattern_discoveries'].values():
        new_terms.update(term for term, freq in matches)
    new_terms.update(term for term, freq in discoveries['novel_high_frequency'])
    
    return new_terms, discoveries

In [74]:
df = ECcontri_Uniprot_enriched.sample(n=15000)

In [75]:
text_columns = ['enzyme_class', 'pathways', 'hierarchy', 'metals_consolidated',
            'corrosion_mechanisms', 'functional_categories',  'corrosion_keyword_groups', 'corrosion_synergies', 'organic_processes'
        ]
analyzer = BiologicalTermDiscovery()
discoveries = analyzer.generate_discovery_report(df, real_terms, text_columns)
analyzer.print_discovery_report(discoveries)

🔬 BIOLOGICAL TERM DISCOVERY REPORT 🔬
📊 Analysis Parameters:
   • Minimum frequency: 10
   • Minimum novelty threshold: 100
   • Top terms per pattern: 3

📈 Discovery Statistics:
   • Total unique terms found: 577
   • Real terms baseline: 143
   • Focused patterns analyzed: 46
   • High-quality pattern matches: 28
   • Novel high-frequency terms: 2

🧬 BIOLOGICAL DISCOVERIES BY CATEGORY:

  🔹 ENZYMES:
     • disease (962x) [pattern: new_ase_terms]
     • oxygenases (284x) [pattern: new_oxy_variants]
     • phospholipase (12x) [pattern: new_ase_terms]

  🔹 METABOLIC PROCESSES:
     • metabolism (42201x) [pattern: new_met_variants]
     • nitrogen_metabolism (796x) [pattern: new_nitr_variants]

⭐ TOP NOVEL HIGH-FREQUENCY TERMS:
   • transfer (23090x)
   • hydrogen (12785x)


In [31]:
discoveries

{'discovery_summary': {'total_unique_terms': 10781,
  'real_terms_baseline': 221,
  'focused_patterns_used': 60,
  'pattern_matches_found': 111,
  'novel_high_freq_terms': 1,
  'columns_processed': ['Genus',
   'protein_name',
   'enzyme_names',
   'enzyme_class',
   'pathways',
   'hierarchy',
   'metals_consolidated',
   'corrosion_mechanisms',
   'functional_categories',
   'corrosion_keyword_groups',
   'corrosion_synergies',
   'organic_processes'],
  'real_categories': ['iron',
   'manganese',
   'copper',
   'nickel',
   'cobalt',
   'magnesium',
   'calcium',
   'Mo',
   'V5+',
   'Al3+',
   'Cr3+',
   'zinc',
   'sodium',
   'potassium',
   'selenium',
   'lead',
   'arsenic',
   'mercury',
   'phosphate',
   'nitrate',
   'nitrite',
   'chloride',
   'sulfate',
   'sulfide',
   'thiosulfate',
   'oxygen',
   'hydrogen',
   'organics',
   'o2_consumption',
   'iron_metabolism',
   'sulfur_metabolism',
   'h2_consumption',
   'direct_eet',
   'carbon_metabolism',
   'indirect_e

In [32]:
analyzer

# Dictionary Refinement

**Dictionary Refinement and Validation for Corrosion-Related Functional Annotation**
To support downstream analysis and scoring in our corrosion microbiome framework, we constructed and iteratively refined a master annotation dictionary (global_terms) composed of multiple biologically meaningful categories (e.g., metal_terms, corrosion_mechanisms, pathway_categories, functional_categories). These dictionaries initially included a comprehensive, theory-driven list of potential terms derived from domain knowledge and literature.

However, during integration with bioinformatic annotation data, it became clear that many terms lacked empirical support in our dataset. To resolve this, we implemented a data-driven refinement workflow as follows:

Step 1: Validation of Terms Against Enriched Data
A custom validation script compared the theoretical dictionary entries against the actual protein annotation fields (e.g., EC numbers, enzyme names, pathways) from our enriched dataset. This yielded a list of real_terms — annotation terms empirically supported in our system.

Step 2: Hierarchical Consolidation of Dictionary Structure
The global_terms structure was then refined using a hybrid strategy:

Minimum viable subcategories: Subcategories (e.g., specific corrosion mechanisms like direct_eet or galvanic_corrosion) were only retained if they contained sufficient real-world support. Sparse subcategories with low or no empirical evidence were eliminated or merged to reduce fragmentation.

Maximum term depth: Within each retained subcategory, we aimed to maximize the number of valid child terms (i.e., biological keywords, gene or enzyme names) to ensure rich annotation coverage.

Semantic reallocation: Unused terms from deprecated categories such as organic_processes and corrosion_keyword_groups were manually reclassified into valid categories where conceptually appropriate (e.g., terms like quorum sensing were reassigned to functional_categories).

This balance of data-driven filtering and semantic grouping led to a revised version of global_terms, maintaining a biologically coherent structure while aligning with the annotation reality of our dataset.

Step 3: Scoring Category Reduction
For downstream scoring and modeling, we focused only on three high-confidence, high-coverage categories:

metal_terms

corrosion_synergies

functional_categories

Other categories (e.g., corrosion_mechanisms, pathway_categories) were retained for network analysis and visualization, but excluded from scoring due to redundancy or sparsity.

# smart_consolidate_terms

In [62]:
def smart_consolidate_terms(global_terms_list, real_terms):
    """
    Match real_terms to global_terms structure.
    - Retains global_terms structure.
    - Adds unmatched but valid terms to the correct top-level category, in a 'miscellaneous' subcategory.
    - Keeps functional_category scores and justification intact.
    - Collects truly unrecognized terms in manual_review.
    
    Args:
        global_terms_list: List of tuples like [('metal_terms', metal_dict), ...]
        real_terms: Dict {subcat: [terms]} from validation script

    Returns:
        consolidated: Updated global_terms-like dict with only valid real_terms
    """
    from collections import defaultdict
    import copy

    # Start from a deep copy of the base terms
    consolidated = {
        'metal_terms': defaultdict(list),
        'corrosion_synergies': defaultdict(list),
        'functional_categories': defaultdict(lambda: {'terms': [], 'score': 1.0}),
        'corrosion_mechanisms': defaultdict(list),
        'pathway_categories': defaultdict(list),
        'manual_review': defaultdict(list)
    }

    # Preserve all subcategories from global_terms
    term_index = {}  # term_lower → (cat, subcat, score)

    for category_name, cat_dict in global_terms_list:
        for subcat, value in cat_dict.items():
            if isinstance(value, dict) and 'terms' in value:
                score = value.get('score', 1.0)
                for term in value['terms']:
                    term_index[term.lower()] = (category_name, subcat, score)
                    consolidated[category_name][subcat] = {
                        'terms': copy.deepcopy(value['terms']),
                        'score': score,
                        'justification': value.get('justification', '')
                    }
            elif isinstance(value, list):
                for term in value:
                    term_index[term.lower()] = (category_name, subcat, None)
                consolidated[category_name][subcat] = copy.deepcopy(value)

    # Reallocate real terms into this structure
    for subcat, terms in real_terms.items():
        for term in terms:
            tkey = term.lower()
            if tkey in term_index:
                cat, existing_subcat, score = term_index[tkey]

                if cat == 'functional_categories':
                    if term not in consolidated[cat][existing_subcat]['terms']:
                        consolidated[cat][existing_subcat]['terms'].append(term)
                else:
                    if term not in consolidated[cat][existing_subcat]:
                        consolidated[cat][existing_subcat].append(term)

            else:
                # Try to infer best category
                likely_cat = (
                    'functional_categories' if 'ase' in tkey or 'eet' in tkey else
                    'metal_terms' if any(m in tkey for m in ['fe', 'cu', 'zn', 'mn']) else
                    'corrosion_synergies' if '-' in tkey else
                    'pathway_categories', 'corrosion_mechanisms'
                )

                if likely_cat == 'functional_categories':
                    consolidated[likely_cat]['miscellaneous']['terms'].append(term)
                elif likely_cat == 'metal_terms':
                    consolidated[likely_cat]['miscellaneous'].append(term)
                elif likely_cat == 'corrosion_synergies':
                    consolidated[likely_cat]['miscellaneous'].append(term)
                elif likely_cat == 'pathway_categories':
                    consolidated[likely_cat]['miscellaneous'].append(term)
                elif likely_cat == 'corrosion_mechanisms':
                    consolidated[likely_cat]['corrosion_mechanisms'].append(term)
                else:
                    consolidated['manual_review'][subcat].append(term)

    return consolidated


In [64]:
consolidated = smart_consolidate_terms([
    ('metal_terms', cs.metal_terms),
    ('corrosion_mechanisms', cs.corrosion_mechanisms),
    ('pathway_categories', cs.pathway_categories),
    ('corrosion_synergies', cs.corrosion_synergies),
    ('functional_categories', cs.functional_categories)
], real_terms)



In [65]:
consolidated

{'metal_terms': defaultdict(list,
             {'iron': ['Fe2+',
               'Fe3+',
               'iron',
               'ferrous',
               'ferric',
               'heme',
               'iron-sulfur',
               'rust',
               'ochre',
               'iron oxide',
               'iron precipitation',
               'siderophore',
               'ferritin'],
              'manganese': ['Mn2+',
               'manganese',
               'mn',
               'manganous',
               'manganic',
               'manganese oxidation',
               'manganese oxide',
               'MnO2'],
              'copper': ['Cu+',
               'Cu2+',
               'copper',
               'cupric',
               'cuprous',
               'copper oxide',
               'copper corrosion'],
              'nickel': ['Ni2+',
               'nickel',
               'nickelous',
               'nickel oxidation',
               'nickel reduction'],
              'cobalt':